# Multi-Region Air Pollution Prediction\n\nUse this notebook in Google Colab to prepare the combined DCR dataset and train the forecasting model. Upload the four station zip files when prompted.

In [ ]:
!python --version\n!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib openpyxl xlrd pyxlsb

## Project Files\n\nIf you opened this notebook from the repository, the scripts are already available. If you are starting from a blank Colab runtime, uncomment the clone cell and replace the GitHub URL with your repository URL.

In [ ]:
# !git clone https://github.com/YOUR_USERNAME/Air_Pollution_Prediction.git\n# %cd Air_Pollution_Prediction\n\nfrom pathlib import Path\nassert Path('scripts/prepare_combined_dataset.py').exists(), 'Run this notebook from the project root or clone the repo first.'

## Upload Raw DCR Zips

In [ ]:
from google.colab import files\n\nuploaded = files.upload()\nzip_files = [name for name in uploaded if name.lower().endswith('.zip')]\nzip_files

## Build the Combined Dataset

In [ ]:
zip_args = ' '.join([f'--zip "{name}"' for name in zip_files])\n!python scripts/prepare_combined_dataset.py {zip_args}

In [ ]:
import pandas as pd\n\nsummary = pd.read_csv('data/processed/all_regions_dataset_summary.csv')\nsummary

In [ ]:
combined = pd.read_csv('data/processed/all_regions_combined.csv', parse_dates=['date_time'], low_memory=False)\nprint(combined.shape)\ncombined.head()

## Train Forecasting Models\n\nThe fast run proves the pipeline. For a stronger final run, increase `--n-estimators` and `--max-depth`.

In [ ]:
!python scripts/train_air_quality_models.py --granularity quarter_hourly --n-estimators 80 --max-depth 14 --max-samples 0.25 --n-jobs -1

In [ ]:
metrics = pd.read_csv('outputs/air_quality_models/reports/metrics_overall.csv')\nmetrics[metrics['split'] == 'test']

In [ ]:
from IPython.display import Image, display\n\nfor image_path in [\n    'outputs/air_quality_models/plots/metric_comparison.png',\n    'outputs/air_quality_models/plots/best_strategy_rmse_heatmap.png',\n    'outputs/air_quality_models/plots/region_prediction_timeseries.png',\n]:\n    if Path(image_path).exists():\n        display(Image(image_path))

## Final Report\n\nOpen `outputs/air_quality_models/reports/summary_report.md` for the model configuration, split sizes, test metrics, and interpretation.